# Review — 3_crewai (100% local, with Ollama)

Review notebook of the **design patterns** covered in the CrewAI reference projects
(`researcher` → `engineering_team`), rewritten to run entirely on **free Ollama models**
(no OpenAI / Serper / Pushover keys) and **inside a notebook** instead of `crewai run`.

## Setup

```bash
ollama pull llama3.2           # small model for plain text agents
ollama pull qwen3:8b           # reliable tool-calling / structured-output model
ollama pull nomic-embed-text   # embeddings for memory (block 8)
OLLAMA_CONTEXT_LENGTH=16384 ollama serve   # default 4096 ctx is too small for crew prompts + tool results
pip install "crewai==1.14.4" ddgs ollama   # ddgs = free web search, replaces SerperDevTool
```

> **Expect slowness.** CrewAI prompts are long and `qwen3:8b` thinks before answering: a full
> run of this notebook took ~1 hour on an Apple Silicon laptop. Run one block at a time.
> Always read the verbose log: small models sometimes *skip a tool* and answer from memory.

> **Notebook vs project.** The course builds each crew as a `crewai create crew` project
> (YAML config + `@CrewBase` class + `crewai run`). Here the *same* objects (`Agent`, `Task`,
> `Crew`) are built directly in Python so every pattern fits in one cell. Block 3 shows the
> 1:1 mapping back to the project layout.

## Index of macro-blocks

| # | Block | Pattern | Source project |
|---|-------|---------|----------------|
| 1 | LLM | `provider/model` string → `LLM` (LiteLLM-style naming) | all |
| 2 | Agent + Task + Crew | role/goal/backstory, description/expected_output, `{inputs}` | `researcher` |
| 3 | YAML ↔ `@CrewBase` | `config=` dict is exactly what the decorators build | all |
| 4 | One agent, many tasks + judge | Same agent reused, `output_file` per task | `debate` |
| 5 | Tools + explicit context | `@tool` (free web search), `context=[task]` | `financial_researcher` |
| 6 | Structured outputs | `output_pydantic=BaseModel` | `stock_picker` |
| 7 | Hierarchical process | `manager_agent` delegates; tool with side-effect | `stock_picker` |
| 8 | Memory | `Memory(llm, embedder)` + `memory=` on the crew | `stock_picker` |
| 9 | Coding agent | Sandbox tools (write / run file), no tool cache | `coder` |
| 10 | Engineering team | Lead → backend → tests, `context=[...]` chain | `engineering_team` |

Skipped (need extra infra / paid keys): `SerperDevTool`, Pushover, CrewAI tracing upload,
Context7 MCP (see `6_mcp`), the Gradio frontend engineer.

In [ ]:
import os
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"   # no span upload (would fail offline anyway)
os.environ["OTEL_SDK_DISABLED"] = "true"

import json, shutil, subprocess, requests
from datetime import datetime
from pathlib import Path
from pydantic import BaseModel, Field
from IPython.display import Markdown, display
from crewai import LLM, Agent, Task, Crew, Process
from crewai.tools import tool

OLLAMA = "http://localhost:11434"
CHAT_LLM = LLM(model="ollama/llama3.2", base_url=OLLAMA)   # plain text agents
TOOL_LLM = LLM(model="ollama/qwen3:8b", base_url=OLLAMA)   # tools, structured output, manager

def check_model(name):
    try:
        models = {m["id"] for m in requests.get(f"{OLLAMA}/v1/models", timeout=3).json().get("data", [])}
    except Exception:
        print("Ollama unreachable on localhost:11434 — make sure 'ollama serve' is running.")
        return False
    if not any(m == name or m.startswith(name + ":") for m in models):
        print(f"Model '{name}' not found. Run: ollama pull {name}")
        return False
    return True

check_model("llama3.2") and check_model("qwen3:8b") and check_model("nomic-embed-text")

## Block 1 — LLM: one string per model

**Pattern:** CrewAI names models as `provider/model` (the LiteLLM convention):
`openai/gpt-5.4-mini`, `anthropic/...`, `ollama/qwen3:8b`. In a project you write it in
`agents.yaml` (`llm: ollama/qwen3:8b`) and put the endpoint in `.env`
(`MODEL=ollama/llama3.2`, `API_BASE=http://localhost:11434`). In code you build an `LLM`
object — same string, plus `base_url`.

> **Gotcha (from my coursework):** `llm:` is an **agent** field. Put it in `agents.yaml`;
> a stray `llm:` under a task in `tasks.yaml` is ignored and the agent silently falls back to
> the default model (OpenAI → fails without a key).

In [ ]:
print(CHAT_LLM.call("In one sentence: what is a crew of AI agents?"))

## Block 2 — Agent + Task + Crew (the `researcher` pattern)

Three building blocks:

1. **Agent** = `role` + `goal` + `backstory` (+ `llm`, `tools`). Together they become the system prompt.
2. **Task** = `description` + `expected_output` + the `agent` that does it (+ `output_file`).
3. **Crew** = agents + tasks + `process`. `Process.sequential` runs tasks in order and passes each
   task's output as **context** to the next one automatically.

`{placeholders}` in any text are filled from `crew.kickoff(inputs={...})` — that's how the
same crew is reused for different topics.

In [ ]:
researcher = Agent(
    role="{topic} Senior Data Researcher",
    goal="Uncover cutting-edge developments in {topic}",
    backstory="You're a seasoned researcher with a knack for uncovering the latest developments in {topic}. "
              "Known for your ability to find the most relevant information and present it in a clear and concise manner.",
    llm=CHAT_LLM,
)
reporting_analyst = Agent(
    role="{topic} Reporting Analyst",
    goal="Create detailed reports based on {topic} data analysis and research findings",
    backstory="You're a meticulous analyst with a keen eye for detail, known for turning complex data into clear reports.",
    llm=CHAT_LLM,
)

research_task = Task(
    description="Conduct a thorough research about {topic}. Make sure you find any interesting and relevant "
                "information given the current year is {current_year}.",
    expected_output="A list with 10 bullet points of the most relevant information about {topic}",
    agent=researcher,
)
reporting_task = Task(
    description="Review the context you got and expand each topic into a full section for a report.",
    expected_output="A fully fledged report with the main topics, each with a full section of information. "
                    "Formatted as markdown without '```'",
    agent=reporting_analyst,
    output_file="output/report.md",
)

crew = Crew(agents=[researcher, reporting_analyst], tasks=[research_task, reporting_task],
            process=Process.sequential, verbose=True)

result = crew.kickoff(inputs={"topic": "AI Agent frameworks", "current_year": str(datetime.now().year)})
display(Markdown(result.raw))

In [ ]:
# The result object keeps every task's output, not only the last one
for t in result.tasks_output:
    print(f"{t.agent.strip()} → {len(t.raw)} chars")
print(result.token_usage)

## Block 3 — YAML config ↔ `@CrewBase` (how the project maps to the code above)

`crewai create crew my_crew` generates:

```
src/my_crew/config/agents.yaml   → role / goal / backstory / llm per agent
src/my_crew/config/tasks.yaml    → description / expected_output / agent / context / output_file
src/my_crew/crew.py              → @CrewBase class: @agent, @task, @crew methods
src/my_crew/main.py              → run(): MyCrew().crew().kickoff(inputs={...})
```

```python
@CrewBase
class Researcher():
    @agent
    def researcher(self) -> Agent:
        return Agent(config=self.agents_config['researcher'], tools=[...])   # YAML + code-only args
    @task
    def research_task(self) -> Task:
        return Task(config=self.tasks_config['research_task'], output_pydantic=...)
    @crew
    def crew(self) -> Crew:
        return Crew(agents=self.agents, tasks=self.tasks, process=Process.sequential)
```

**Pattern:** YAML holds the *prompts* (what you iterate on), Python holds the *wiring*
(tools, Pydantic models, process). `@agent`/`@task` collect themselves into `self.agents` /
`self.tasks` **in definition order** — that order is the sequential order. Under the hood each
entry is just passed as `config=`:

In [ ]:
import yaml

agents_yaml = yaml.safe_load("""
researcher:
  role: >
    {topic} Senior Data Researcher
  goal: >
    Uncover cutting-edge developments in {topic}
  backstory: >
    You're a seasoned researcher with a knack for uncovering the latest developments in {topic}.
""")

# What @agent + config=self.agents_config['researcher'] does, minus the decorator
yaml_researcher = Agent(config=agents_yaml["researcher"], llm=CHAT_LLM)
print(yaml_researcher.role, "|", yaml_researcher.goal)

## Block 4 — One agent, many tasks + a judge (the `debate` pattern)

**Pattern:** agents are reusable — the **same** `debater` runs both `propose` and `oppose`.
A second agent (`judge`) sees both outputs (sequential context) and decides.
Each task writes its own `output_file`, so every intermediate step is inspectable.

This is the LLM-as-judge from `1_foundations`, now expressed as a crew.

In [ ]:
debater = Agent(
    role="A compelling debater",
    goal="Present a clear argument either in favor of or against the motion. The motion is: {motion}.",
    backstory="You're an experienced debater with a knack for giving concise but convincing arguments.",
    llm=TOOL_LLM,   # llama3.2 refused to argue one side of this motion
)
judge = Agent(
    role="Decide the winner of the debate based on the arguments presented",
    goal="Given arguments for and against this motion: {motion}, decide which side is more convincing.",
    backstory="You are a fair judge who decides purely on the merits of the arguments, not your own views.",
    llm=TOOL_LLM,
)

propose = Task(description="You are proposing the motion: {motion}. Come up with a clear, convincing argument in favor.",
               expected_output="Your argument in favor of the motion, concise.", agent=debater, output_file="output/propose.md")
oppose = Task(description="You are in opposition to the motion: {motion}. Come up with a clear, convincing argument against.",
              expected_output="Your argument against the motion, concise.", agent=debater, output_file="output/oppose.md")
decide = Task(description="Review the arguments presented by the debaters and decide which side is more convincing.",
              expected_output="Your decision on which side is more convincing, and why.", agent=judge, output_file="output/decide.md")

debate = Crew(agents=[debater, judge], tasks=[propose, oppose, decide], process=Process.sequential)
result = debate.kickoff(inputs={"motion": "There needs to be strict laws to regulate LLMs"})
display(Markdown(result.raw))

## Block 5 — Tools + explicit context (the `financial_researcher` pattern)

**Pattern:** give an agent `tools=[...]`; the agent loop decides when to call them.
Two ways to build a custom tool:

- `@tool("Name")` on a function — schema from the signature + docstring (like `@function_tool` in `2_openai`)
- subclass `BaseTool` with `args_schema` + `_run` (the `custom_tool.py` template)

The reference uses `SerperDevTool` (Google search, API key). Here: a free `@tool` on DuckDuckGo (`ddgs`).

`context=[research_task]` makes the dependency **explicit** — required when tasks are not
adjacent, or when a task needs several upstream outputs (see Block 10).

In [ ]:
from ddgs import DDGS

@tool("Web Search")
def web_search(query: str) -> str:
    """Search the web and return the top results (title, snippet, url) as text."""
    hits = DDGS().text(query, max_results=5)
    return "\n".join(f"- {h['title']}: {h['body']} ({h['href']})" for h in hits)

fin_researcher = Agent(
    role="Senior Financial Researcher for {company}",
    goal="Research latest company information, news and potential for {company}, up to date as of {current_date}.",
    backstory="You're a seasoned financial researcher with a talent for finding the most relevant information.",
    llm=TOOL_LLM, tools=[web_search],
)
analyst = Agent(
    role="Market Analyst and Report writer focused on {company}",
    goal="Analyze company {company} and create a comprehensive, well-structured report.",
    backstory="You're a meticulous analyst who extracts meaningful insights and communicates them in well crafted reports.",
    llm=TOOL_LLM,
)

fin_research = Task(
    description="Research company {company}: current status, historical performance, challenges and opportunities, "
                "recent news, future outlook. Use your Web Search tool for every section, do not answer from memory. Information MUST be up to date as of {current_date}.",
    expected_output="A research document with well-organized sections covering all the requested aspects of {company}.",
    agent=fin_researcher,
)
analysis = Task(
    description="Analyze the research findings and write a report on {company}: executive summary, key information, "
                "trends, market outlook (not to be used for trading decisions).",
    expected_output="A polished markdown report with executive summary, main sections, and conclusion.",
    agent=analyst,
    context=[fin_research],          # explicit dependency (YAML: context: [research_task])
    output_file="output/financial_report.md",
)

result = Crew(agents=[fin_researcher, analyst], tasks=[fin_research, analysis]).kickoff(
    inputs={"company": "Tesla", "current_date": str(datetime.now().date())})
display(Markdown(result.raw))

## Block 6 — Structured outputs (the `stock_picker` pattern, part 1)

**Pattern:** `Task(output_pydantic=Model)` → CrewAI asks the model for JSON matching the schema
and gives back a **Python object** in `result.pydantic`. The next task still receives it as
text context, but *you* get typed data — same idea as `output_type=` in `2_openai`.

The Pydantic class docstring and every `Field(description=...)` are part of the prompt.

In [ ]:
class TrendingCompany(BaseModel):
    """A company that is in the news and attracting attention"""
    name: str = Field(description="Company name")
    ticker: str = Field(description="Stock ticker symbol")
    reason: str = Field(description="Reason this company is trending in the news")

class TrendingCompanyList(BaseModel):
    """List of multiple trending companies that are in the news"""
    companies: list[TrendingCompany] = Field(description="List of companies trending in the news")

trending_finder = Agent(
    role="Financial News Analyst that finds trending companies in {sector} as of {current_date}",
    goal="Read the latest news, then find 2-3 companies that are trending in the news for further research.",
    backstory="You are a market expert with a knack for picking out the most interesting companies based on latest news.",
    llm=TOOL_LLM, tools=[web_search],
)
find_trending = Task(
    description="Find the top trending companies in the news in {sector} as of {current_date} using your Web Search tool. "
                "Only report real companies found in the search results. Don't pick companies you picked before.",
    expected_output="A list of trending companies in {sector}",
    agent=trending_finder,
    output_pydantic=TrendingCompanyList,
    output_file="output/trending_companies.json",
)

inputs = {"sector": "Technology", "current_date": str(datetime.now().date())}
result = Crew(agents=[trending_finder], tasks=[find_trending]).kickoff(inputs=inputs)
for c in result.pydantic.companies:   # a real Python object, not text
    print(f"{c.ticker:6} {c.name}: {c.reason}")

## Block 7 — Hierarchical process (the `stock_picker` pattern, part 2)

**Pattern:** `Process.hierarchical` + a `manager_agent` (with `allow_delegation=True`, **no tools**).
Instead of running tasks in a fixed order, the manager decides *who* does *what*, delegating
through built-in "delegate work" / "ask question" tools — CrewAI's version of *agents as tools*
from `2_openai`. Alternative: `manager_llm=...` and CrewAI builds a default manager.

Also here: a custom tool with a **side-effect** (Pushover notification in the reference → `print`).

> Small local models are hit-and-miss as managers (they may loop on delegation or skip steps).
> Sequential (Blocks 2-6) is the reliable default; hierarchical is where a stronger model pays off.

In [ ]:
class TrendingCompanyResearch(BaseModel):
    """Detailed research on a company"""
    name: str = Field(description="Company name")
    market_position: str = Field(description="Current market position and competitive analysis")
    future_outlook: str = Field(description="Future outlook and growth prospects")
    investment_potential: str = Field(description="Investment potential and suitability for investment")

class TrendingCompanyResearchList(BaseModel):
    """A list of detailed research on all the companies"""
    research_list: list[TrendingCompanyResearch] = Field(description="Comprehensive research on all trending companies")

@tool("Send Push Notification")
def send_push_notification(message: str) -> str:
    """Use this tool to send a push notification to the user."""
    print(f"[PUSH] {message}")   # replaces the Pushover HTTP call
    return "Push notification sent"

stock_researcher = Agent(
    role="Senior Financial Researcher",
    goal="Given details of trending companies in the news, provide comprehensive analysis of each in a report.",
    backstory="You are a financial expert with a proven track record of deeply analyzing hot companies.",
    llm=TOOL_LLM, tools=[web_search],
)
stock_picker = Agent(
    role="Stock Picker from Research",
    goal="Given a list of researched companies, select the best one for investment. Notify the user, then provide a report.",
    backstory="You're a meticulous, skilled financial analyst with a proven track record of equity selection.",
    llm=TOOL_LLM, tools=[send_push_notification],
)
manager = Agent(
    role="Manager",
    goal="You are a skilled project manager who delegates tasks in order to pick the best company for investment.",
    backstory="You are an experienced and highly effective project manager who delegates tasks to the right people.",
    llm=TOOL_LLM, allow_delegation=True,
)

research_trending = Task(
    description="Given a list of trending companies, provide detailed analysis of each company by searching online.",
    expected_output="A report containing detailed analysis of each company",
    agent=stock_researcher, context=[find_trending], output_pydantic=TrendingCompanyResearchList,
)
pick_best = Task(
    description="Analyze the research findings and pick the best company for investment. Send a push notification "
                "with the decision and 1 sentence rationale. Then respond with a report on why you chose it.",
    expected_output="The chosen company and why it was chosen; the companies that were not selected and why.",
    agent=stock_picker, context=[research_trending], output_file="output/decision.md",
)

stock_crew = Crew(
    agents=[trending_finder, stock_researcher, stock_picker],   # the manager is NOT in this list
    tasks=[find_trending, research_trending, pick_best],
    process=Process.hierarchical, manager_agent=manager, verbose=True,
)
result = stock_crew.kickoff(inputs=inputs)
display(Markdown(result.raw))

## Block 8 — Memory

**Pattern:** in CrewAI 1.x memory is a single `Memory` object: an LLM analyses what to store
(scope, categories, importance) and an **embedder** makes it searchable in a local vector DB
(LanceDB). Pass it as `Crew(memory=...)` and agents get *remember* / *recall* tools automatically.
Memory persists **across runs** — in `stock_picker` that's what makes "don't pick the same company twice" work.

Out of the box it uses OpenAI for both the LLM and the embeddings → here both point to Ollama.

In [ ]:
from crewai.memory import Memory

memory = Memory(
    llm=TOOL_LLM,
    embedder={"provider": "ollama", "config": {"model_name": "nomic-embed-text"}},
    storage="./memory",   # on-disk folder, survives kernel restarts
)

# Direct use: the same calls the agents make through their memory tools
memory.remember("Last run the stock picker selected NVIDIA (NVDA) in the Technology sector.")
for match in memory.recall("Which company was picked before?"):
    print(round(match.score, 2), match.record.content)

In [ ]:
# In a crew: the finder gets recall/remember tools and can avoid previous picks (small models don't always use them)
remembering_crew = Crew(agents=[trending_finder], tasks=[find_trending], memory=memory)
result = remembering_crew.kickoff(inputs=inputs)
print([c.name for c in result.pydantic.companies])

## Block 9 — Coding agent (the `coder` pattern)

**Pattern:** an agent that writes code and **runs it**, through four plain `@tool`s over a
sandbox folder: list / read / write / run. The run happens in an **ephemeral Docker container**
(`docker run --rm -v sandbox:/workspace`), so generated code never touches your machine.

Two details worth remembering:

- `tool.cache_function = lambda *a, **k: False` — CrewAI caches tool results by arguments;
  "run solution.py" must re-execute after each edit, so caching is turned off.
- Return **stdout + stderr + exit code** (the `engineering_team` version): errors are how the agent fixes its code.

No Docker? Set `ALLOW_LOCAL_EXEC = True` to run with your local Python instead (only for code you'd run yourself).

In [ ]:
SANDBOX = Path("sandbox").resolve()
SANDBOX.mkdir(exist_ok=True)
HAS_DOCKER = shutil.which("docker") is not None
ALLOW_LOCAL_EXEC = False   # fallback when Docker is missing: runs LLM-written code unsandboxed

@tool("List Sandbox Files")
def list_sandbox_files() -> str:
    """List the filenames currently in the sandbox directory."""
    return "\n".join(sorted(p.name for p in SANDBOX.iterdir())) or "The sandbox is empty."

@tool("Read Sandbox File")
def read_sandbox_file(filename: str) -> str:
    """Read and return the text contents of a file in the sandbox directory."""
    path = SANDBOX / Path(filename).name
    return path.read_text() if path.is_file() else f"No such file in the sandbox: {filename}"

@tool("Write Sandbox File")
def write_sandbox_file(filename: str, content: str) -> str:
    """Write text to a file in the sandbox directory, replacing any existing file with the same name."""
    (SANDBOX / Path(filename).name).write_text(content)
    return f"Wrote {len(content)} characters to {filename}."

@tool("Run Sandbox Python File")
def run_sandbox_python(filename: str) -> str:
    """Run a Python file from the sandbox directory and return exit code, stdout and stderr."""
    name = Path(filename).name
    if HAS_DOCKER:
        cmd = ["docker", "run", "--rm", "-v", f"{SANDBOX}:/workspace", "-w", "/workspace", "python:3.13-slim", "python", name]
    elif ALLOW_LOCAL_EXEC:
        cmd = ["python", name]
    else:
        return "Cannot run code: Docker is not available."
    try:
        r = subprocess.run(cmd, cwd=SANDBOX, capture_output=True, text=True, timeout=120)
    except subprocess.TimeoutExpired:
        return "The script timed out after 120 seconds."
    return f"Exit code: {r.returncode}\n--- stdout ---\n{r.stdout or '(empty)'}\n--- stderr ---\n{r.stderr or '(empty)'}"

sandbox_tools = [list_sandbox_files, read_sandbox_file, write_sandbox_file, run_sandbox_python]
for t in sandbox_tools:
    t.cache_function = lambda *a, **k: False   # always re-execute

print("Docker:", HAS_DOCKER, "| local fallback:", ALLOW_LOCAL_EXEC)

In [ ]:
coder = Agent(
    role="Python Developer",
    goal="Use your sandbox tools to achieve this assignment: {assignment}. "
         "Write a python file in the sandbox directory, then run it and check the output.",
    backstory="You're a seasoned python developer with a knack for writing clean, efficient code.",
    llm=TOOL_LLM, tools=sandbox_tools,
)
coding_task = Task(
    description="Use your sandbox tools to write and run python code to achieve this: {assignment}",
    expected_output="A summary of what you did to achieve the assignment and the final result.",
    agent=coder, output_file="output/solution.md",
)

assignment = ("Write a python program to calculate the first 1,000,000 terms of this series, "
              "multiplying the total by 4: 1 - 1/3 + 1/5 - 1/7 + ...")
result = Crew(agents=[coder], tasks=[coding_task]).kickoff(inputs={"assignment": assignment})
display(Markdown(result.raw))

## Block 10 — Engineering team (the `engineering_team` pattern, trimmed)

**Pattern:** a sequential crew of *specialists* sharing one sandbox, where each task lists the
upstream tasks it needs in `context=[...]`:

`engineering_lead (design, no code) → backend_engineer (code) → test_engineer (unittest, fix until green)`

- The lead only produces a **design** (signatures, no code) — cheap, and it steers everyone else.
- Engineers get **constraints in the goal** (stdlib only, one directory, no UI) — small models need them.
- The test engineer receives `context=[code_task, design_task]`: both the spec and the implementation.

> With `qwen3:8b` the backend engineer sometimes *describes* the code instead of calling the
> write tool — check the sandbox listing at the end. `gpt-oss:20b` (or a paid model) follows
> multi-step tool instructions more reliably; the reference uses `gpt-5.5` for the lead.

Trimmed vs. the reference: no Gradio frontend engineer, no Context7 MCP server
(`Agent(mcps=["https://mcp.context7.com/mcp"])` — the reference also needs a monkey-patch for
hyphenated tool names in 1.14.4; MCP is covered in `6_mcp`).

In [ ]:
requirements = """
A simple Account class for a trading simulation: create an account, deposit and withdraw funds,
record buying or selling shares (symbol, quantity), and report holdings, portfolio value and
profit/loss versus the initial deposit. Prevent negative balances, buying more than affordable,
and selling shares not owned. Use get_share_price(symbol) with fixed prices for AAPL, TSLA, GOOGL.
"""

lead = Agent(
    role="Engineering Lead",
    goal="Given high level requirements, design the system for a backend engineer and a test engineer: "
         "module, class, method signatures. Do not write any code. Everything is one file, Python stdlib only.",
    backstory="You're a seasoned engineering lead with a knack for writing clear and concise designs.",
    llm=TOOL_LLM,
)
backend = Agent(
    role="Backend Engineer",
    goal="Use your sandbox tools to write the python module described by the engineering lead. "
         "Only the Python standard library is available. No UI code.",
    backstory="You're a seasoned python engineer who follows the design instructions carefully.",
    llm=TOOL_LLM, tools=sandbox_tools,
)
tester = Agent(
    role="Test Engineer",
    goal="Use your sandbox tools to write unit tests with the built-in unittest module for the backend module, "
         "run them, fix any defects and rerun until they pass.",
    backstory="You're a seasoned QA engineer who writes great unit tests for python code.",
    llm=TOOL_LLM, tools=sandbox_tools,
)

design_task = Task(description="Prepare a detailed design in markdown for these requirements: {requirements}. "
                               "Signatures only, no code. The module file is accounts.py.",
                   expected_output="A markdown design with the module, class and method signatures.",
                   agent=lead, output_file="sandbox/design.md")
code_task = Task(description="Write accounts.py implementing the design. Requirements: {requirements}",
                 expected_output="accounts.py written to the sandbox and checked by running it. "
                                 "IMPORTANT: use your Write Sandbox File tool, don't just print the code.",
                 agent=backend, context=[design_task])
test_task = Task(description="Write test_accounts.py with unittest for accounts.py, run it, fix errors until all tests pass.",
                 expected_output="A summary of the unit test results. "
                                 "IMPORTANT: write test_accounts.py with your tools and run it with Run Sandbox Python File.",
                 agent=tester, context=[code_task, design_task], output_file="sandbox/test_summary.md")

team = Crew(agents=[lead, backend, tester], tasks=[design_task, code_task, test_task], process=Process.sequential)
result = team.kickoff(inputs={"requirements": requirements})
display(Markdown(result.raw))
print(list_sandbox_files.run())

## Pattern recap

| Pattern | Key idea |
|---|---|
| `provider/model` string | One naming scheme for every model; `llm:` lives on the **agent** |
| Agent / Task / Crew | Agent = who (role, goal, backstory); Task = what (description, expected output); Crew = how (process) |
| `{inputs}` interpolation | Same crew, different runs: placeholders filled at `kickoff(inputs=...)` |
| YAML + `@CrewBase` | Prompts in YAML, wiring in Python; `config=` is the bridge |
| Reused agent + judge | One agent on many tasks; a judge agent reads their outputs |
| `@tool` / `BaseTool` | Custom tools from a docstring; free search replaces `SerperDevTool` |
| `context=[task]` | Explicit data flow between non-adjacent tasks |
| `output_pydantic` | Typed Python object instead of text (`result.pydantic`) |
| Hierarchical process | A manager agent delegates instead of a fixed order (needs a strong model) |
| `Memory` | LLM-curated, embedding-searchable, persists across runs |
| Sandbox tools | Write + run code in Docker; disable tool cache; return stderr too |
| Specialist team | Design → code → test, each with the context it needs |

Compared to `2_openai`: CrewAI is more **opinionated** (roles, tasks, processes are first-class),
the OpenAI Agents SDK is closer to the raw agent loop. To review the original paid version,
see `3_crewai/reference/`; for the from-scratch loop behind all of this, `1_foundations/_review_1_foundations.ipynb`.